# CSE284 Final Project - Comparing Global Ancestry Analysis Methods on 1000 Genomes Phase 3 Data

In [1]:
from pathlib import Path
import os

candidate = Path.cwd()
repo_root = None
for path in [candidate, *candidate.parents]:
    if (path / "admixture.sh").exists() and (path / "structure.sh").exists():
        repo_root = path
        break

if repo_root is None:
    raise FileNotFoundError("Cannot locate repo root containing admixture.sh and structure.sh")

os.chdir(repo_root)
print("Current working directory:", os.getcwd())

Current working directory: /home/wjl/Population_Structure_Modeling


## Data Exploration

In [2]:
from pathlib import Path
import pandas as pd

repo = Path.cwd()
print(f"Repo root: {repo}")

# 1) Sample metadata (1000 Genomes)
igsr_path = repo / "1000Genomes/igsr_samples.tsv"
igsr_df = pd.read_csv(igsr_path, sep="\t")
print("\n[igsr_samples.tsv]")
print("shape:", igsr_df.shape)
print("columns:", list(igsr_df.columns))
print("dtypes (first 10):")
print(igsr_df.dtypes.head(10))
print("\nhead:")
display(igsr_df.head(3))

Repo root: /home/wjl/Population_Structure_Modeling

[igsr_samples.tsv]
shape: (4978, 9)
columns: ['Sample name', 'Sex', 'Biosample ID', 'Population code', 'Population name', 'Superpopulation code', 'Superpopulation name', 'Population elastic ID', 'Data collections']
dtypes (first 10):
Sample name              object
Sex                      object
Biosample ID             object
Population code          object
Population name          object
Superpopulation code     object
Superpopulation name     object
Population elastic ID    object
Data collections         object
dtype: object

head:


,Sample name,Sex,Biosample ID,Population code,Population name,Superpopulation code,Superpopulation name,Population elastic ID,Data collections
0,HG00271,male,SAME123417,FIN,Finnish,EUR,European Ancestry,FIN,"1000 Genomes on GRCh38,1000 Genomes 30x on GRC..."
1,HG00276,female,SAME123424,FIN,Finnish,EUR,European Ancestry,FIN,"1000 Genomes on GRCh38,1000 Genomes 30x on GRC..."
2,HG00288,female,SAME1839246,FIN,Finnish,EUR,European Ancestry,FIN,"1000 Genomes on GRCh38,1000 Genomes 30x on GRC..."


## Data Preprocessing

In [3]:
from pathlib import Path
import os
import re
import shlex
import subprocess

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import FileLink, display


REPO = Path.cwd()
IGSR_PATH = REPO / "1000Genomes/igsr_samples.tsv"
K_PLOT_LIST = [2, 4, 6, 8]


def plink_file(prefix_path: Path, extension: str) -> Path:
    return Path(f"{prefix_path}{extension}")


def merged_dataset_prefix(out_dir: str, prefix: str, ld_pruned: bool = True) -> Path:
    suffix = ".pruned" if ld_pruned else ""
    return Path(out_dir) / f"{prefix}_ALL{suffix}"


def show_log_link(message: str, log_path: str | Path) -> None:
    log_path = Path(log_path)
    print(f"{message}: {log_path}")
    display(FileLink(str(log_path)))


def run_preprocess_if_needed(
    out_dir: str,
    prefix: str,
    sample_info: str = "1000Genomes/igsr_samples.tsv",
    chr_start: int = 1,
    chr_end: int = 22,
    maf: float = 0.01,
    ld_window: int = 50,
    ld_step: int = 10,
    ld_r2: float = 0.1,
    ld_pruned: bool = True,
    force: bool = False,
    log_path: str | Path | None = None,
) -> Path:
    dataset_prefix = merged_dataset_prefix(out_dir, prefix, ld_pruned=ld_pruned)
    required_paths = [
        plink_file(dataset_prefix, ".bed"),
        plink_file(dataset_prefix, ".bim"),
        plink_file(dataset_prefix, ".fam"),
    ]
    if all(path.exists() for path in required_paths) and not force:
        print(f"Found shared preprocessed data at {dataset_prefix}, skip preprocess.")
        return dataset_prefix

    cmd = [
        "bash",
        "preprocess.sh",
        "--out-dir",
        out_dir,
        "--prefix",
        prefix,
        "--sample-info",
        sample_info,
        "--chr-start",
        str(chr_start),
        "--chr-end",
        str(chr_end),
        "--maf",
        str(maf),
        "--ld-window",
        str(ld_window),
        "--ld-step",
        str(ld_step),
        "--ld-r2",
        str(ld_r2),
    ]
    if not ld_pruned:
        cmd.append("--skip-ld-prune")

    log_path = Path(log_path or Path(out_dir) / "preprocess.log")
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", shlex.join(cmd))
    with log_path.open("w") as log_file:
        subprocess.run(cmd, check=True, stdout=log_file, stderr=subprocess.STDOUT)
    show_log_link("Preprocess finished. Log", log_path)
    return dataset_prefix


def build_structure_ind_file(fam_path: str | Path, ind_file_path: str | Path) -> Path:
    igsr = pd.read_csv(IGSR_PATH, sep="\t")
    phase3 = igsr[
        igsr["Data collections"].str.contains(
            "1000 Genomes phase 3 release",
            case=False,
            na=False,
        )
    ]
    sample_to_pop = dict(zip(phase3["Sample name"], phase3["Population code"]))

    fam = pd.read_csv(
        fam_path,
        sep=r"\s+",
        header=None,
        names=["FID", "IID", "PID", "MID", "SEX", "PHENO"],
        dtype=str,
    )
    assert (fam["FID"] == fam["IID"]).all(), "FID != IID: --double-id may not have been used during preprocessing"

    fam["pop"] = fam["IID"].map(sample_to_pop)
    pop_order = sorted(fam["pop"].dropna().unique())
    pop_to_order = {population: i + 1 for i, population in enumerate(pop_order)}
    fam["order"] = fam["pop"].map(pop_to_order)

    ind_file_path = Path(ind_file_path)
    ind_file_path.parent.mkdir(parents=True, exist_ok=True)
    fam[["IID", "pop", "order"]].to_csv(
        ind_file_path,
        sep="\t",
        header=False,
        index=False,
    )
    print(
        f"Ind file written: {len(fam)} individuals, {len(pop_order)} populations, "
        f"missing_pop={fam['pop'].isna().sum()} -> {ind_file_path}"
    )
    return ind_file_path


def run_method_script(
    script_name: str,
    *,
    out_dir: str,
    input_bed: str | Path,
    prefix: str,
    log_path: str | Path,
    ind_file: str | Path | None = None,
) -> None:
    env = os.environ.copy()
    env.update(
        {
            "OUT_DIR": str(out_dir),
            "INPUT_BED": str(input_bed),
            "PREFIX": str(prefix),
        }
    )
    if ind_file is not None:
        env["IND_FILE"] = str(ind_file)

    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    cmd = ["bash", script_name]
    print("Running:", shlex.join(cmd))
    print(f"  OUT_DIR={out_dir}")
    print(f"  INPUT_BED={input_bed}")
    print(f"  PREFIX={prefix}")
    if ind_file is not None:
        print(f"  IND_FILE={ind_file}")
    with log_path.open("w") as log_file:
        subprocess.run(cmd, check=True, env=env, stdout=log_file, stderr=subprocess.STDOUT)
    show_log_link(f"{script_name} finished. Log", log_path)


def show_experiment_sanity(
    dataset_prefix: str | Path,
    admixture_dir: str | Path,
    structure_dir: str | Path,
    structure_prefix: str,
) -> None:
    dataset_prefix = Path(dataset_prefix)
    fam_path = plink_file(dataset_prefix, ".fam")
    bim_path = plink_file(dataset_prefix, ".bim")

    fam_cols = ["FID", "IID", "PID", "MID", "SEX", "PHENO"]
    fam_df = pd.read_csv(fam_path, sep=r"\s+", header=None, names=fam_cols)
    print(f"[{fam_path.name}]")
    print("shape:", fam_df.shape)
    display(fam_df.head(3))

    bim_cols = ["CHR", "SNP", "CM", "BP", "A1", "A2"]
    bim_df = pd.read_csv(bim_path, sep=r"\s+", header=None, names=bim_cols)
    print(f"\n[{bim_path.name}]")
    print("shape:", bim_df.shape)
    display(bim_df.head(3))

    q_files = sorted(Path(admixture_dir).glob(f"{dataset_prefix.name}.*.Q"))
    q_summary = []
    for qf in q_files:
        match = re.search(r"\.([0-9]+)\.Q$", qf.name)
        if match is None:
            continue
        q_df = pd.read_csv(qf, sep=r"\s+", header=None)
        q_summary.append(
            {
                "file": qf.name,
                "K": int(match.group(1)),
                "n_samples": q_df.shape[0],
                "n_components": q_df.shape[1],
                "matches_fam_rows": q_df.shape[0] == fam_df.shape[0],
            }
        )
    print("\n[ADMIXTURE Q files]")
    display(pd.DataFrame(q_summary).sort_values("K"))

    meanq_dir = Path(structure_dir) / f"{structure_prefix}_ALL"
    meanq_files = sorted(meanq_dir.glob("fS_run_K.*.meanQ"))
    meanq_summary = []
    for meanq_file in meanq_files:
        match = re.search(r"K\.([0-9]+)\.meanQ$", meanq_file.name)
        if match is None:
            continue
        meanq_df = pd.read_csv(meanq_file, sep=r"\s+", header=None)
        meanq_summary.append(
            {
                "file": meanq_file.name,
                "K": int(match.group(1)),
                "n_samples": meanq_df.shape[0],
                "n_components": meanq_df.shape[1],
                "matches_fam_rows": meanq_df.shape[0] == fam_df.shape[0],
            }
        )
    print("\n[fastSTRUCTURE meanQ files]")
    display(pd.DataFrame(meanq_summary).sort_values("K"))


def plot_membership_panels(
    *,
    fam_path: str | Path,
    data_path_template: str,
    title_prefix: str,
    k_list: list[int] | None = None,
) -> None:
    if k_list is None:
        k_list = K_PLOT_LIST

    sampleinfo = pd.read_csv(IGSR_PATH, sep="\t")
    sample_to_pop = dict(zip(sampleinfo["Sample name"], sampleinfo["Population code"]))
    samples = pd.read_csv(fam_path, sep=r"\s+", header=None, usecols=[0]).iloc[:, 0].tolist()
    populations = [sample_to_pop.get(sample, "NA") for sample in samples]

    fig = plt.figure(figsize=(16, 12))
    for plot_index, k_value in enumerate(k_list, start=1):
        ax = fig.add_subplot(len(k_list), 1, plot_index)
        data = pd.read_csv(data_path_template.format(K=k_value), sep=r"\s+", header=None)
        cols = list(data.columns)
        data["sample"] = samples
        data["pop"] = populations
        data = data.sort_values(["pop"] + cols)
        data[cols].plot.bar(stacked=True, ax=ax, width=1)
        ax.spines["right"].set_visible(False)
        ax.spines["top"].set_visible(False)
        ax.yaxis.set_ticks_position("left")
        ax.xaxis.set_ticks_position("bottom")

        xticklabels = []
        current_population = ""
        for population in data["pop"].tolist():
            if population == current_population:
                xticklabels.append("")
            else:
                xticklabels.append(population)
                current_population = population
        if plot_index < len(k_list):
            ax.set_xticklabels([])
        else:
            ax.set_xticklabels(xticklabels, rotation=90, fontsize=12)
        ax.set_title(f"{title_prefix}, K={k_value}")
    fig.tight_layout()


BASELINE_DATASET_PREFIX = run_preprocess_if_needed(
    out_dir="dump/common",
    prefix="common",
    force=False,
)
print(f"Shared preprocess is completed above: {BASELINE_DATASET_PREFIX}")

Running: bash preprocess.sh --out-dir dump/common --prefix common --sample-info 1000Genomes/igsr_samples.tsv --chr-start 1 --chr-end 22 --maf 0.01 --ld-window 50 --ld-step 10 --ld-r2 0.1


--2026-03-11 16:25:59--  https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/reference/human_g1k_v37.fasta.gz
Resolving ftp.1000genomes.ebi.ac.uk (ftp.1000genomes.ebi.ac.uk)... 193.62.193.167
Connecting to ftp.1000genomes.ebi.ac.uk (ftp.1000genomes.ebi.ac.uk)|193.62.193.167|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 892331003 (851M) [application/x-gzip]
Saving to: ‘1000Genomes/reference/human_g1k_v37.fasta.gz’

     0K .......... .......... .......... .......... ..........  0%  151K 96m22s
    50K .......... .......... .......... .......... ..........  0%  315K 71m16s
   100K .......... .......... .......... .......... ..........  0% 2.49M 49m24s
   150K .......... .......... .......... .......... ..........  0%  310K 48m46s
   200K .......... .......... .......... .......... ..........  0% 12.8M 39m14s
   250K .......... .......... .......... .......... ..........  0% 5.22M 33m9s
   300K .......... .......... .......... .......... ..........  0% 5.02M 2

KeyboardInterrupt: 

In [ ]:
# Generate the baseline per-individual population file for structure_threader (--ind format)

BASELINE_IND_FILE = build_structure_ind_file(
    plink_file(BASELINE_DATASET_PREFIX, ".fam"),
    "dump/structure/structure_ind_file.tsv",
)

## Run Experiments Using Bash Files

In [ ]:
run_method_script(
    "admixture.sh",
    out_dir="dump/admixture",
    input_bed=plink_file(BASELINE_DATASET_PREFIX, ".bed"),
    prefix="admixture",
    log_path="dump/admixture/admixture_pipeline.log",
)

In [ ]:
run_method_script(
    "structure.sh",
    out_dir="dump/structure",
    input_bed=plink_file(BASELINE_DATASET_PREFIX, ".bed"),
    prefix="structure",
    ind_file=BASELINE_IND_FILE,
    log_path="dump/structure/structure_pipeline.log",
)

### Additional Experiment: No LD Pruning

In [ ]:
NO_LD_DATASET_PREFIX = run_preprocess_if_needed(
    out_dir="dump/common_no_ld",
    prefix="common_no_ld",
    maf=0.01,
    ld_pruned=False,
    force=False,
)
NO_LD_IND_FILE = build_structure_ind_file(
    plink_file(NO_LD_DATASET_PREFIX, ".fam"),
    "dump/structure_no_ld/structure_no_ld_ind_file.tsv",
)

run_method_script(
    "admixture.sh",
    out_dir="dump/admixture_no_ld",
    input_bed=plink_file(NO_LD_DATASET_PREFIX, ".bed"),
    prefix="admixture_no_ld",
    log_path="dump/admixture_no_ld/admixture_pipeline.log",
)
run_method_script(
    "structure.sh",
    out_dir="dump/structure_no_ld",
    input_bed=plink_file(NO_LD_DATASET_PREFIX, ".bed"),
    prefix="structure_no_ld",
    ind_file=NO_LD_IND_FILE,
    log_path="dump/structure_no_ld/structure_pipeline.log",
)

In [ ]:
show_experiment_sanity(
    NO_LD_DATASET_PREFIX,
    admixture_dir="dump/admixture_no_ld",
    structure_dir="dump/structure_no_ld",
    structure_prefix="structure_no_ld",
)

plot_membership_panels(
    fam_path=plink_file(NO_LD_DATASET_PREFIX, ".fam"),
    data_path_template="dump/admixture_no_ld/common_no_ld_ALL.{K}.Q",
    title_prefix="ADMIXTURE (No LD prune)",
)
plot_membership_panels(
    fam_path=plink_file(NO_LD_DATASET_PREFIX, ".fam"),
    data_path_template="dump/structure_no_ld/structure_no_ld_ALL/fS_run_K.{K}.meanQ",
    title_prefix="fastSTRUCTURE (No LD prune)",
)

### Additional Experiment: MAF = 0.05

In [ ]:
MAF005_DATASET_PREFIX = run_preprocess_if_needed(
    out_dir="dump/common_maf005",
    prefix="common_maf005",
    maf=0.05,
    ld_pruned=True,
    force=False,
)
MAF005_IND_FILE = build_structure_ind_file(
    plink_file(MAF005_DATASET_PREFIX, ".fam"),
    "dump/structure_maf005/structure_maf005_ind_file.tsv",
)

run_method_script(
    "admixture.sh",
    out_dir="dump/admixture_maf005",
    input_bed=plink_file(MAF005_DATASET_PREFIX, ".bed"),
    prefix="admixture_maf005",
    log_path="dump/admixture_maf005/admixture_pipeline.log",
)
run_method_script(
    "structure.sh",
    out_dir="dump/structure_maf005",
    input_bed=plink_file(MAF005_DATASET_PREFIX, ".bed"),
    prefix="structure_maf005",
    ind_file=MAF005_IND_FILE,
    log_path="dump/structure_maf005/structure_pipeline.log",
)

In [ ]:
show_experiment_sanity(
    MAF005_DATASET_PREFIX,
    admixture_dir="dump/admixture_maf005",
    structure_dir="dump/structure_maf005",
    structure_prefix="structure_maf005",
)

plot_membership_panels(
    fam_path=plink_file(MAF005_DATASET_PREFIX, ".fam"),
    data_path_template="dump/admixture_maf005/common_maf005_ALL.pruned.{K}.Q",
    title_prefix="ADMIXTURE (MAF=0.05)",
)
plot_membership_panels(
    fam_path=plink_file(MAF005_DATASET_PREFIX, ".fam"),
    data_path_template="dump/structure_maf005/structure_maf005_ALL/fS_run_K.{K}.meanQ",
    title_prefix="fastSTRUCTURE (MAF=0.05)",
)

### Post-Experiment Output Sanity Checks

In [ ]:
from pathlib import Path
import re
import pandas as pd

repo = Path.cwd()

# 0) Shared PLINK FAM/BIM checks
fam_path = repo / "dump/common/common_ALL.pruned.fam"
fam_cols = ["FID", "IID", "PID", "MID", "SEX", "PHENO"]
fam_df = pd.read_csv(fam_path, sep=r"\s+", header=None, names=fam_cols)
print("[common_ALL.pruned.fam]")
print("shape:", fam_df.shape)
display(fam_df.head(3))

bim_path = repo / "dump/common/common_ALL.pruned.bim"
bim_cols = ["CHR", "SNP", "CM", "BP", "A1", "A2"]
bim_df = pd.read_csv(bim_path, sep=r"\s+", header=None, names=bim_cols)
print("\n[common_ALL.pruned.bim]")
print("shape:", bim_df.shape)
display(bim_df.head(3))

# 1) ADMIXTURE Q files: K, rows, columns
print("\n[ADMIXTURE Q files]")
q_files = sorted((repo / "dump/admixture").glob("common_ALL.pruned.*.Q"))
q_summary = []
for qf in q_files:
    m = re.search(r"\.([0-9]+)\.Q$", qf.name)
    if not m:
        continue
    k = int(m.group(1))
    q_df = pd.read_csv(qf, sep=r"\s+", header=None)
    q_summary.append({
        "file": qf.name,
        "K": k,
        "n_samples": q_df.shape[0],
        "n_components": q_df.shape[1],
        "matches_fam_rows": q_df.shape[0] == fam_df.shape[0],
    })

q_summary_df = pd.DataFrame(q_summary).sort_values("K")
display(q_summary_df)

# 2) fastSTRUCTURE meanQ files: K, rows, columns
print("\n[fastSTRUCTURE meanQ files]")
meanq_dir = repo / "dump/structure/structure_ALL"
meanq_files = sorted(meanq_dir.glob("fS_run_K.*.meanQ"))
meanq_summary = []
for mf in meanq_files:
    m = re.search(r"K\.([0-9]+)\.meanQ$", mf.name)
    if not m:
        continue
    k = int(m.group(1))
    m_df = pd.read_csv(mf, sep=r"\s+", header=None)
    meanq_summary.append({
        "file": mf.name,
        "K": k,
        "n_samples": m_df.shape[0],
        "n_components": m_df.shape[1],
        "matches_fam_rows": m_df.shape[0] == fam_df.shape[0],
    })

meanq_summary_df = pd.DataFrame(meanq_summary).sort_values("K")
display(meanq_summary_df)

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, f1_score,
)

repo = Path.cwd()
EPS = 1e-12
SUPER_POPS = ["AFR", "AMR", "EAS", "EUR", "SAS"]

def load_q_matrix(path):
    q = pd.read_csv(path, sep=r"\s+", header=None).to_numpy(dtype=float)
    q = np.clip(q, EPS, None)
    q /= q.sum(axis=1, keepdims=True)
    return q


def load_iids_from_fam(path):
    fam = pd.read_csv(path, sep=r"\s+", header=None, usecols=[1], names=["IID"], dtype=str)
    return fam["IID"].tolist()


adm_q_files = sorted((repo / "dump/admixture").glob("common_ALL.pruned.*.Q"))
fst_q_files = sorted((repo / "dump/structure/structure_ALL").glob("fS_run_K.*.meanQ"))

adm_map = {int(re.search(r"\.([0-9]+)\.Q$", p.name).group(1)): p for p in adm_q_files}
fst_map = {int(re.search(r"K\.([0-9]+)\.meanQ$", p.name).group(1)): p for p in fst_q_files}
K_common = sorted(set(adm_map) & set(fst_map))

shared_fam = repo / "dump/common/common_ALL.pruned.fam"
common_iids = load_iids_from_fam(shared_fam)

adm_Q = {k: load_q_matrix(adm_map[k]) for k in K_common}
fst_Q = {k: load_q_matrix(fst_map[k]) for k in K_common}

igsr = pd.read_csv(repo / "1000Genomes/igsr_samples.tsv", sep="\t")
sample_col = "Sample name" if "Sample name" in igsr.columns else "Sample"
superpop_col = next(
    c for c in ["Superpopulation code", "Superpopulation", "Super Population"]
    if c in igsr.columns
)
sample_to_superpop = dict(zip(igsr[sample_col].astype(str), igsr[superpop_col].astype(str)))
superpops = [sample_to_superpop.get(iid, np.nan) for iid in common_iids]

print(f"Common K values: {K_common}")
print(f"Common samples:  {len(common_iids)}")
print(f"Labeled samples: {sum(pd.notna(s) for s in superpops)}")

## Quantitative Comparison on Time and Space Complexity

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

repo = Path.cwd()
adm_metrics_path = repo / "dump/admixture/metrics.tsv"
fst_metrics_path = repo / "dump/structure/metrics.tsv"

adm = pd.read_csv(adm_metrics_path, sep="\t")
fst = pd.read_csv(fst_metrics_path, sep="\t")

metrics = pd.concat([adm, fst], ignore_index=True)

# Keep successful runs only
metrics = metrics[metrics["exit_code"] == 0].copy()

# Numeric conversion (NA -> NaN)
for col in ["elapsed_sec", "user_cpu_sec", "sys_cpu_sec", "max_rss_kb"]:
    metrics[col] = pd.to_numeric(metrics[col], errors="coerce")

# Keep fit step with valid numeric K for method-to-method comparison
fit = metrics[(metrics["step"] == "fit")].copy()
fit["K_num"] = pd.to_numeric(fit["K"], errors="coerce")
fit = fit.dropna(subset=["K_num"])
fit["K_num"] = fit["K_num"].astype(int)

# Method naming for plotting
method_map = {"admixture": "ADMIXTURE", "faststructure": "fastSTRUCTURE"}
fit["method_label"] = fit["method"].map(method_map).fillna(fit["method"])

fit["total_cpu_sec"] = fit["user_cpu_sec"].fillna(0) + fit["sys_cpu_sec"].fillna(0)
fit["cpu_efficiency"] = fit["total_cpu_sec"] / fit["elapsed_sec"].replace(0, np.nan)
fit["max_rss_mb"] = fit["max_rss_kb"] / 1024

# Wide table for side-by-side comparison
pivot = fit.pivot_table(
    index="K_num",
    columns="method_label",
    values=["elapsed_sec", "max_rss_mb", "cpu_efficiency"],
    aggfunc="mean",
)

# Add relative metrics (fastSTRUCTURE / ADMIXTURE) where both exist
if {"ADMIXTURE", "fastSTRUCTURE"}.issubset(pivot["elapsed_sec"].columns):
    time_ratio = pivot[("elapsed_sec", "fastSTRUCTURE")] / pivot[("elapsed_sec", "ADMIXTURE")]
    mem_ratio = pivot[("max_rss_mb", "fastSTRUCTURE")] / pivot[("max_rss_mb", "ADMIXTURE")]
    ratio_df = pd.DataFrame({
        "time_ratio_fastSTRUCTURE_over_ADMIXTURE": time_ratio,
        "memory_ratio_fastSTRUCTURE_over_ADMIXTURE": mem_ratio,
    })
    display(ratio_df.round(3))

print("\n[Fit-step summary by method]")
summary = fit.groupby("method_label").agg(
    runs=("K_num", "count"),
    k_min=("K_num", "min"),
    k_max=("K_num", "max"),
    elapsed_mean_sec=("elapsed_sec", "mean"),
    elapsed_sum_sec=("elapsed_sec", "sum"),
    rss_mean_mb=("max_rss_mb", "mean"),
    rss_peak_mb=("max_rss_mb", "max"),
    cpu_eff_mean=("cpu_efficiency", "mean"),
).round(3)
display(summary)

print("\n[Per-K detailed table]")
detail = fit[["method_label", "K_num", "elapsed_sec", "max_rss_mb", "total_cpu_sec", "cpu_efficiency"]].sort_values(["K_num", "method_label"])
display(detail.round(3))

# ---------- Visualization ----------
# Line plots: over K
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for method_label, g in fit.groupby("method_label"):
    g = g.sort_values("K_num")
    axes[0].plot(g["K_num"], g["elapsed_sec"], marker="o", label=method_label)
    axes[1].plot(g["K_num"], g["max_rss_mb"], marker="o", label=method_label)

axes[0].set_title("Runtime vs K (fit step)")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Elapsed time (sec)")
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Peak memory vs K (fit step)")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Max RSS (MB)")
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

# #  Barplot
# k_values = sorted(fit["K_num"].unique())
# methods = [m for m in ["ADMIXTURE", "fastSTRUCTURE"] if m in fit["method_label"].unique()]
# x = np.arange(len(k_values))
# width = 0.35

# fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
# for idx, m in enumerate(methods):
#     g = fit[fit["method_label"] == m].set_index("K_num").reindex(k_values)
#     offset = (idx - (len(methods)-1)/2) * width
#     axes[0].bar(x + offset, g["elapsed_sec"], width=width, label=m)
#     axes[1].bar(x + offset, g["max_rss_mb"], width=width, label=m)

# axes[0].set_xticks(x, k_values)
# axes[0].set_title("Runtime per K (bar)")
# axes[0].set_xlabel("K")
# axes[0].set_ylabel("Elapsed time (sec)")
# axes[0].grid(axis="y", alpha=0.3)

# axes[1].set_xticks(x, k_values)
# axes[1].set_title("Peak memory per K (bar)")
# axes[1].set_xlabel("K")
# axes[1].set_ylabel("Max RSS (MB)")
# axes[1].grid(axis="y", alpha=0.3)

# for ax in axes:
#     ax.legend()
# plt.tight_layout()
# plt.show()

# Scatter for time-memory tradeoff
plt.figure(figsize=(6, 5))
for method_label, g in fit.groupby("method_label"):
    plt.scatter(g["elapsed_sec"], g["max_rss_mb"], label=method_label, alpha=0.8)
    for _, row in g.iterrows():
        plt.text(row["elapsed_sec"], row["max_rss_mb"], f"K={row['K_num']}", fontsize=8)

plt.title("Time-Memory tradeoff (fit step)")
plt.xlabel("Elapsed time (sec)")
plt.ylabel("Max RSS (MB)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Qualitative Comparison on Global Ancestry Inference Results

### Visulization of Population Structure

You can run our analysis code below to observe something interesting when exploring population structure.

**High-Level Consistency**: Both models successfully identify the major continental clusters that correspond to the 1000 Genomes Project populations. Even as $K$ increases, the "backbone" of the population structure remains stable. For instance, European (EUR) groups such as CEU, GBR, and IBS show high consistency, while ASW and ACB (African Americans/Caribbeans) exhibit complex admixture patterns.

**fastSTRUCTURE with less noise**: fastSTRUCTURE tends to produce **cleaner** looking plots with less background noise. This is because its underlying variational Bayesian framework often forces individuals toward 0 or 1 membership unless there is strong evidence for admixture. In contrast, Admixture model appears more sensitive to subtle gene flow. You can see more **noise** or low-level components across almost all individuals, which often reflects actual historical interbreeding rather than just mathematical artifacts.

**The "Splitter" Effect**: As you move from $K=4$ to $K=8$, you can observe how a single ancestral component in one population (e.g., YRI group) begins to fragment into two distinct sub-components, indicating finer-scale sub-structure within that geographic region.

In [ ]:

import pandas as pd
import os
import matplotlib.pyplot as plt

sampleinfo = pd.read_csv("1000Genomes/igsr_samples.tsv", sep="\t")
sample_to_pop = dict(zip(list(sampleinfo["Sample name"]), list(sampleinfo["Population code"])))
samples = [line.split()[0] for line in open("dump/common/common_ALL.pruned.fam", "r").readlines()]
pops = [sample_to_pop.get(item, "NA") for item in samples]

fig = plt.figure()
fig.set_size_inches((16, 12))

plotind = 1
k_list = [2,4,6,8]
for K in k_list:
    ax = fig.add_subplot(len(k_list), 1, plotind)

    data = pd.read_csv("dump/admixture/common_ALL.pruned.%s.Q" % K, sep=r"\s+", header=None)
    cols = list(data.columns)
    data["sample"] = samples
    data["pop"] = pops
    data = data.sort_values(["pop"] + cols)
    data[cols].plot.bar(stacked=True, ax=ax, width=1)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')

    # Only plot label for first sample in each pop
    xticklabels = []
    currpop = ""
    for i in range(data.shape[0]):
        if data["pop"].values[i] == currpop:
            xticklabels.append("")
        else:
            xticklabels.append(data["pop"].values[i])
            currpop = data["pop"].values[i]
    if plotind < len(k_list):
        ax.set_xticklabels([])
    else:
        ax.set_xticklabels(xticklabels, rotation=90, fontsize=12)
    ax.set_title(f"ADMIXTURE, K={K}")

    plotind += 1
fig.tight_layout()


In [ ]:

import pandas as pd
import os
import matplotlib.pyplot as plt
sampleinfo = pd.read_csv("1000Genomes/igsr_samples.tsv", sep="\t")
sample_to_pop = dict(zip(list(sampleinfo["Sample name"]), list(sampleinfo["Population code"])))
samples = [line.split()[0] for line in open("dump/common/common_ALL.pruned.fam", "r").readlines()]
pops = [sample_to_pop.get(item, "NA") for item in samples]

fig = plt.figure()
fig.set_size_inches((16, 12))

plotind = 1
k_list = [2,4,6,8]
for K in k_list:
    ax = fig.add_subplot(len(k_list), 1, plotind)

    data = pd.read_csv("dump/structure/structure_ALL/fS_run_K.%s.meanQ" % K, sep=r"\s+", header=None)
    cols = list(data.columns)
    data["sample"] = samples
    data["pop"] = pops
    data = data.sort_values(["pop"] + cols)
    data[cols].plot.bar(stacked=True, ax=ax, width=1)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('bottom')

    # Only plot label for first sample in each pop
    xticklabels = []
    currpop = ""
    for i in range(data.shape[0]):
        if data["pop"].values[i] == currpop:
            xticklabels.append("")
        else:
            xticklabels.append(data["pop"].values[i])
            currpop = data["pop"].values[i]
    if plotind < len(k_list):
        ax.set_xticklabels([])
    else:
        ax.set_xticklabels(xticklabels, rotation=90, fontsize=12)
    ax.set_title(f"fastSTRUCTURE, K={K}")

    plotind += 1
fig.tight_layout()


## Quantitative Comparison on Global Ancestry Inference Results

### K-value Evaluation

To determine the optimal number of ancestral populations ($K$), we evaluated two different metrics across a range of $K$ values (from 2 to 10). Choosing the "right" $K$ involves identifying the point where model fit improves significantly before hitting diminishing returns.

#### Methodology and Metrics
- ADMIXTURE (Cross-Validation Error): 
We utilized 10-fold cross-validation. In this metric, a lower CV error indicates a better-supported $K$ value, as it suggests the model has better predictive power.
- fastSTRUCTURE (Marginal Likelihood): We measured the model's marginal likelihood. In this case, a higher (less negative) value indicates a more probable $K$ for the given dataset.

#### Results and Interpretation
Based on the generated plots, we observed the following:

- ADMIXTURE: The CV error decreases sharply from $K=2$ to $K=5$. After $K=5$, the rate of decrease slows down significantly, suggesting that additional clusters beyond this point add marginal explanatory power.

- fastSTRUCTURE: The marginal likelihood shows an evident peak and "elbow" at $K=5/6$.

Based on the combined evidence from both tools—specifically the sharp decline in CV error and the plateau in marginal likelihood—we have selected $K = 4, 5, 6, \text{and } 7$ as the candidate values for further population structure analysis. 
<!-- These values capture the primary levels of genetic differentiation without over-partitioning the data. -->

In [ ]:
import matplotlib.pyplot as plt
import re
from pathlib import Path

repo = Path.cwd()

# Load ADMIXTURE CV errors from output file
cv_path = repo / "dump" / "admixture" / "cv_results.txt"
cv_pairs = []
for line in cv_path.read_text().strip().splitlines():
    k_str, cv_str = line.split()
    cv_pairs.append((int(k_str), float(cv_str)))
cv_pairs.sort(key=lambda x: x[0])


def extract_marginal_likelihood(log_path: Path) -> float:
    text = log_path.read_text()
    match = re.search(r"Marginal Likelihood =\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", text)
    if match is None:
        raise ValueError(f"Cannot find marginal likelihood in {log_path}")
    return float(match.group(1))


# Load fastSTRUCTURE marginal likelihoods from per-K logs
ml_pairs = []
for k, _ in cv_pairs:
    log_path = repo / "dump" / "structure" / "structure_ALL" / f"fS_run_K.{k}.log"
    ml_pairs.append((k, extract_marginal_likelihood(log_path)))

k_values = [k for k, _ in cv_pairs]
cv_errors = [value for _, value in cv_pairs]
ml = [value for _, value in ml_pairs]

# Create two subplots (1 row, 2 columns)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ----- Subplot 1: ADMIXTURE CV error -----
axes[0].plot(k_values, cv_errors, marker='o')
axes[0].set_xlabel("K Value")
axes[0].set_ylabel("Cross-Validation Error")
axes[0].set_title("ADMIXTURE CV Error vs K")
axes[0].set_xticks(k_values)
axes[0].grid(True)

# ----- Subplot 2: fastSTRUCTURE Marginal Likelihood -----
axes[1].plot(k_values, ml, marker='o', color='orange')
axes[1].set_xlabel("K Value")
axes[1].set_ylabel("Marginal Likelihood")
axes[1].set_title("fastSTRUCTURE Marginal Likelihood vs K")
axes[1].set_xticks(k_values)
axes[1].grid(True)

plt.tight_layout()
plt.show()

### Superpopulation Hard Classification

The 1000 Genomes project provides 5 **superpopulation labels** (AFR / AMR / EAS / EUR / SAS) for each individual. Although they are not mixture proportion continuous ground truths, they serve as an external reference for evaluating whether the inferred ancestry assignment is consistent with known population structure.

**Approach:**

0. Consider K = 5 as there are 5 superpopulations.
1. For each individual $i$, take the *hard assignment* $\hat{k}_i = \arg\max_k Q_{ik}$.
2. For each component $k$, identify its plurality superpopulation (the super-pop most common among individuals whose hard assignment is $k$).
3. Map each individual's hard-assignment component to its plurality superpopulation label and compare against the real superpopulation label.
4. Report **accuracy** and **macro-F1** only.

In [ ]:

import re
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

SUPER_POPS = ["AFR", "AMR", "EAS", "EUR", "SAS"]
K = 5

_SPLIT = re.compile(r"[,\|;/\s]+")

def parse_superpops(x):
    if x is None or pd.isna(x):
        return []
    parts = [p.strip() for p in _SPLIT.split(str(x).strip()) if p.strip()]
    out = [p for p in parts if p in SUPER_POPS]
    seen, out2 = set(), []
    for p in out:
        if p not in seen:
            out2.append(p); seen.add(p)
    return out2

def hard_assign(Q):
    return np.argmax(Q, axis=1)

def component_to_superpop_multilabel(assignments, true_sets, K):
    comp_map = {}
    for k in range(K):
        m = (assignments == k)
        if not np.any(m):
            comp_map[k] = "NA"
            continue
        counts = pd.Series(0, index=SUPER_POPS, dtype=int)
        for labs in np.array(true_sets, dtype=object)[m]:
            for sp in labs:
                counts[sp] += 1
        comp_map[k] = counts.idxmax() if counts.sum() > 0 else "NA"
    return comp_map

def multilabel_crosstab(true_sets, y_pred):
    rows = []
    for s, pred in zip(true_sets, y_pred):
        for sp in s:
            rows.append((sp, pred))
    df = pd.DataFrame(rows, columns=["True", "Predicted"])
    return pd.crosstab(df["True"], df["Predicted"])

def evaluate_superpop_multilabel(Q, superpops_list, K=5):
    labels_raw = np.asarray(superpops_list, dtype=object)
    true_sets = [parse_superpops(x) for x in labels_raw]
    valid = np.array([len(s) > 0 for s in true_sets], dtype=bool)

    Qv = np.asarray(Q, float)[valid]
    true_sets_v = np.array(true_sets, dtype=object)[valid]

    assigns = hard_assign(Qv)
    comp_map = component_to_superpop_multilabel(assigns, true_sets_v, K)
    y_pred = np.array([comp_map.get(a, "NA") for a in assigns], dtype=object)

    acc = float(np.mean([pred in s for pred, s in zip(y_pred, true_sets_v)]))

    sp_to_idx = {sp: i for i, sp in enumerate(SUPER_POPS)}
    Y_true = np.zeros((len(true_sets_v), len(SUPER_POPS)), dtype=int)
    for i, s in enumerate(true_sets_v):
        for sp in s:
            Y_true[i, sp_to_idx[sp]] = 1

    Y_pred = np.zeros_like(Y_true)
    for i, pred in enumerate(y_pred):
        j = sp_to_idx.get(pred)
        if j is not None:
            Y_pred[i, j] = 1

    macro_f1 = f1_score(Y_true, Y_pred, average="macro", zero_division=0)
    return {"accuracy": acc, "macro_F1": macro_f1}, comp_map, true_sets_v, y_pred

# ---- K=5 only ----
for name, Q in [("ADMIXTURE", adm_Q[5]), ("fastSTRUCTURE", fst_Q[5])]:
    metrics, comp_map, true_sets_v, y_pred = evaluate_superpop_multilabel(Q, superpops, K=5)
    print(f"\n{name} hard-assignment metrics (K=5, multi-label truth): {metrics}")
    print(f"{name} component->superpop mapping:", comp_map)
    print(f"{name} multi-label crosstab:")
    display(multilabel_crosstab(true_sets_v, y_pred))

### Superpopulation Soft Classification

Rather than collapsing $Q$ to a hard label, we can treat each individual's ancestry proportion vector as a **soft probability assignment** over the $K$ inferred components and evaluate how faithfully it captures the known superpopulation label using the multi-class Brier Score:

$$\text{BS} = \frac{1}{N} \sum_{i=1}^{N} \sum_{k=1}^{K} \left( \hat{p}_{ik} - y_{ik} \right)^2 $$

where $y_{ik} \in \{0,1\}$ is the one-hot superpopulation indicator and $\hat{p}_{ik}$ is the ancestry proportion assigned to the component that was matched (via plurality mapping) to superpopulation $k$.

A lower Brier Score means the soft output is better calibrated to the known superpopulation labels. This provides a stricter assessment because it rewards confident correct predictions and penalises uncertain or wrong ones.

In [ ]:
K = 5
EPS = 1e-12

def build_Q5_Y_multilabel(Q, superpops_list, K=5):
    labels_raw = np.asarray(superpops_list, dtype=object)
    true_sets = [parse_superpops(x) for x in labels_raw]
    valid = np.array([len(s) > 0 for s in true_sets], dtype=bool)

    Qv = np.asarray(Q, float)[valid]
    Qv = np.clip(Qv, EPS, None)
    Qv /= Qv.sum(axis=1, keepdims=True)

    true_sets_v = np.array(true_sets, dtype=object)[valid]
    assigns = hard_assign(Qv)
    comp_map = component_to_superpop_multilabel(assigns, true_sets_v, K)

    sp_to_idx = {sp: i for i, sp in enumerate(SUPER_POPS)}

    Q5 = np.zeros((len(true_sets_v), len(SUPER_POPS)), dtype=float)
    for k, sp in comp_map.items():
        j = sp_to_idx.get(sp)
        if j is not None:
            Q5[:, j] += Qv[:, k]

    rs = Q5.sum(axis=1, keepdims=True)
    Q5 /= np.where(rs == 0, 1.0, rs)

    Y = np.zeros_like(Q5)
    for i, s in enumerate(true_sets_v):
        w = 1.0 / len(s)
        for sp in s:
            Y[i, sp_to_idx[sp]] += w

    return Q5, Y, comp_map

def brier(Q5, Y):
    return ((Q5 - Y) ** 2).sum(axis=1).mean()

def per_superpop_brier(Q5, Y):
    out = {}
    for j, sp in enumerate(SUPER_POPS):
        m = Y[:, j] > 0
        out[sp] = ((Q5[m] - Y[m]) ** 2).sum(axis=1).mean() if np.any(m) else np.nan
    return out

def eval_brier_K5(Q, labels):
    Q5, Y, comp_map = build_Q5_Y_multilabel(Q, labels, K=5)
    return brier(Q5, Y), per_superpop_brier(Q5, Y), comp_map

# ---- K=5 only ----
adm_overall, adm_sp, adm_map = eval_brier_K5(adm_Q[5], superpops)
fst_overall, fst_sp, fst_map = eval_brier_K5(fst_Q[5], superpops)

print("\nBrier (K=5, multi-label truth as uniform distribution over labels):")
display(pd.DataFrame([{
    "K": 5,
    "ADMIXTURE": adm_overall,
    "fastSTRUCTURE": fst_overall
}]).round(5).set_index("K"))

print("\nComponent->superpop mapping (multi-label voting):")
print("ADMIXTURE:", adm_map)
print("fastSTRUCTURE:", fst_map)

sp_df = pd.DataFrame({
    "Super-pop": SUPER_POPS,
    "ADMIXTURE": [adm_sp[sp] for sp in SUPER_POPS],
    "fastSTRUCTURE": [fst_sp[sp] for sp in SUPER_POPS],
}).round(5).set_index("Super-pop")
print("\nPer-super-pop Brier (multi-label samples contribute to every class they contain):")
display(sp_df)